In [0]:
from pyspark.sql import functions as F

CAT  = 'northbridge'
S_TX, S_RJ, S_LG = f'{CAT}.silver.transactions', f'{CAT}.silver.transactions_rejected', f'{CAT}.silver.ledger'
DIM  = f'{CAT}.silver.dim_customer'
DQ   = f'{CAT}.ops.dq_results'

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE northbridge.gold.daily_kpis AS
WITH s AS (
  SELECT day, txn_type, count(*) AS txn_count, sum(amount) AS total_amount, avg(amount) AS avg_amount
  FROM northbridge.silver.transactions GROUP BY day, txn_type),
r AS (
  SELECT day, coalesce(type, 'UNKNOWN') AS txn_type, count(*) AS rejected_count
  FROM northbridge.silver.transactions_rejected GROUP BY day, coalesce(type, 'UNKNOWN'))
SELECT coalesce(s.day, r.day) AS day, coalesce(s.txn_type, r.txn_type) AS txn_type,
       coalesce(s.txn_count, 0) AS txn_count, s.total_amount, s.avg_amount,
       coalesce(r.rejected_count, 0) AS rejected_count,
       round(coalesce(r.rejected_count, 0) / (coalesce(s.txn_count, 0) + coalesce(r.rejected_count, 0)), 6) AS rejected_rate
FROM s FULL OUTER JOIN r ON s.day = r.day AND s.txn_type = r.txn_type
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE northbridge.gold.fraud_flags AS
WITH f AS (
  SELECT txn_id, day, txn_type, amount, orig_id, dest_id, is_fraud_label,
         (old_bal_orig > 0 AND new_bal_orig = 0) AS drained,
         (amount >= 200000) AS large_amt,
         (old_bal_dest = 0) AS dest_empty
  FROM northbridge.silver.transactions
  WHERE txn_type IN ('TRANSFER', 'CASH_OUT'))
SELECT txn_id, day, txn_type, amount, orig_id, dest_id, is_fraud_label,
       CASE WHEN large_amt AND dest_empty THEN 'HIGH'
            WHEN dest_empty THEN 'MEDIUM'
            ELSE 'LOW' END AS risk_tier,
       filter(array(IF(drained, 'ACCOUNT_DRAINED', NULL),
                    IF(large_amt, 'LARGE_AMOUNT', NULL),
                    IF(dest_empty, 'DEST_EMPTY', NULL)), x -> x IS NOT NULL) AS reasons,
       array_join(filter(array(IF(drained, 'ACCOUNT_DRAINED', NULL),
                    IF(large_amt, 'LARGE_AMOUNT', NULL),
                    IF(dest_empty, 'DEST_EMPTY', NULL)), x -> x IS NOT NULL), ',') AS reason_list
FROM f WHERE drained
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE northbridge.gold.reconciliation AS
SELECT coalesce(g.txn_id, l.txn_id) AS txn_id, coalesce(g.day, l.day) AS day,
       g.amount AS gateway_amount, l.ledger_amount,
       l.ledger_amount - g.amount AS amount_diff,
       CASE WHEN l.txn_id IS NULL THEN 'MISSING_IN_LEDGER'
            WHEN g.txn_id IS NULL THEN 'MISSING_IN_GATEWAY'
            ELSE 'AMOUNT_MISMATCH' END AS recon_status
FROM northbridge.silver.transactions g
FULL OUTER JOIN northbridge.silver.ledger l ON g.txn_id = l.txn_id
WHERE l.txn_id IS NULL OR g.txn_id IS NULL OR g.amount <> l.ledger_amount
""")

display(spark.sql("""
  SELECT recon_status, count(*) AS rows,
         round(100 * count(*) / (SELECT count(*) FROM northbridge.silver.transactions), 3) AS pct_of_txns
  FROM northbridge.gold.reconciliation GROUP BY recon_status"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE northbridge.gold.fact_transactions AS
SELECT t.txn_id, t.day, t.step,
       timestampadd(HOUR, t.step - 1, TIMESTAMP'2024-01-01 00:00:00') AS txn_ts,
       t.txn_type, t.amount, t.orig_id,
       c.city AS orig_city, c.account_status AS orig_account_status,
       t.dest_id, t.old_bal_orig, t.new_bal_orig, t.old_bal_dest, t.new_bal_dest,
       t.is_fraud_label, f.txn_id IS NOT NULL AS is_flagged, f.reason_list AS flag_reasons, f.risk_tier AS risk_tier
FROM northbridge.silver.transactions t
LEFT JOIN northbridge.silver.dim_customer c
  ON t.orig_id = c.customer_id
 AND timestampadd(HOUR, t.step - 1, TIMESTAMP'2024-01-01 00:00:00') >= c.valid_from
 AND timestampadd(HOUR, t.step - 1, TIMESTAMP'2024-01-01 00:00:00') <  c.valid_to
LEFT JOIN northbridge.gold.fraud_flags f ON t.txn_id = f.txn_id
""")

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE northbridge.gold.customer_360 AS
WITH cur AS (
  SELECT customer_id, city, account_status FROM northbridge.silver.dim_customer WHERE is_current),
hist AS (
  SELECT customer_id, count(*) AS versions FROM northbridge.silver.dim_customer GROUP BY customer_id),
beh AS (
  SELECT orig_id AS customer_id, count(*) AS txn_count, sum(amount) AS total_sent,
         avg(amount) AS avg_sent, max(day) AS last_txn_day,
         sum(CASE WHEN is_flagged THEN 1 ELSE 0 END) AS flagged_txns,
         sum(CASE WHEN risk_tier = 'HIGH' THEN 1 ELSE 0 END) AS high_txns
  FROM northbridge.gold.fact_transactions GROUP BY orig_id)
SELECT c.customer_id, c.city, c.account_status, h.versions AS profile_versions,
       coalesce(b.txn_count, 0) AS txn_count, b.total_sent, b.avg_sent, b.last_txn_day,
       coalesce(b.flagged_txns, 0) AS flagged_txns,
       CASE WHEN coalesce(b.high_txns, 0) >= 1 THEN 'HIGH'
            WHEN coalesce(b.flagged_txns, 0) >= 1 THEN 'MEDIUM' ELSE 'LOW' END AS risk_level,
       CASE WHEN b.total_sent >= 500000 THEN 'HIGH_VALUE'
            WHEN b.total_sent >= 50000 THEN 'MID_VALUE' ELSE 'STANDARD' END AS value_segment
FROM cur c
LEFT JOIN hist h ON c.customer_id = h.customer_id
LEFT JOIN beh b ON c.customer_id = b.customer_id
""")

In [0]:
total_fraud = spark.table(S_TX).filter('is_fraud_label = 1').count()
total_txn   = spark.table(S_TX).count()
base_rate   = total_fraud / total_txn

base = (spark.table(S_TX).select('txn_id', 'is_fraud_label')
        .join(spark.table(f'{CAT}.gold.fraud_flags').select('txn_id', 'risk_tier'), 'txn_id', 'left'))

views = [('TIER_HIGH', ['HIGH']), ('TIER_MEDIUM', ['MEDIUM']), ('TIER_LOW', ['LOW']),
         ('QUEUE_HIGH', ['HIGH']), ('QUEUE_HIGH_MEDIUM', ['HIGH', 'MEDIUM']),
         ('QUEUE_ALL', ['HIGH', 'MEDIUM', 'LOW'])]

rows = []
for name, tiers in views:
    flagged = F.coalesce(F.col('risk_tier').isin(tiers), F.lit(False))
    r = base.agg(
        F.sum(F.when(flagged, 1).otherwise(0)).alias('n'),
        F.sum(F.when(flagged & (F.col('is_fraud_label') == 1), 1).otherwise(0)).alias('tp')).first()
    n, tp = int(r.n), int(r.tp)
    prec = tp / n if n else None
    rows.append((name, n, tp, n - tp, total_fraud - tp, prec,
                 tp / total_fraud, prec / base_rate if prec else None))

scores = (spark.createDataFrame(rows,
          'rule string, flagged long, tp long, fp long, fn long, precision double, recall double, lift double')
          .withColumn('scored_at', F.current_timestamp()))
spark.sql(f'DROP TABLE IF EXISTS {CAT}.gold.fraud_rule_scores')
scores.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{CAT}.gold.fraud_rule_scores')
display(scores)

In [0]:
def log_dq(name, ok, n, notes=''):
    (spark.createDataFrame([(name, 'PASS' if ok else 'FAIL', n, notes)],
         'check_name string, status string, row_count long, notes string')
      .withColumn('run_ts', F.current_timestamp())
      .select('check_name', 'run_ts', 'status', 'row_count', 'notes')
      .write.mode('append').saveAsTable(DQ))

n_silver = spark.table(S_TX).count()
fact = spark.table(f'{CAT}.gold.fact_transactions')
n_fact, n_fact_ids = fact.count(), fact.select('txn_id').distinct().count()
n_no_city = fact.filter('orig_city IS NULL').count()
n_c360  = spark.table(f'{CAT}.gold.customer_360').count()
n_cur   = spark.table(DIM).filter('is_current').count()
k = spark.table(f'{CAT}.gold.daily_kpis').agg(F.sum('txn_count'), F.sum('rejected_count')).first()
n_rej = spark.table(S_RJ).count()

log_dq('gold_fact_rows_eq_silver', n_fact == n_silver and n_fact == n_fact_ids, n_fact)
log_dq('gold_fact_all_customers_matched', n_no_city == 0, n_no_city)
log_dq('gold_customer_360_eq_current_dim', n_c360 == n_cur, n_c360)
log_dq('gold_kpis_tie_to_silver', k[0] == n_silver and k[1] == n_rej, n_silver)

display(spark.table(DQ).orderBy(F.col('run_ts').desc()).limit(8))

In [0]:
from pyspark.sql import functions as F

S_TX = 'northbridge.silver.transactions'

total_fraud = spark.table(S_TX).filter('is_fraud_label = 1').count()

def score_rule(name, cond):
    r = spark.table(S_TX).agg(
        F.sum(F.when(cond & (F.col('is_fraud_label') == 1), 1).otherwise(0)).alias('tp'),
        F.sum(F.when(cond & (F.col('is_fraud_label') == 0), 1).otherwise(0)).alias('fp')).first()
    tp, fp = int(r.tp), int(r.fp)
    return (name, tp, fp, total_fraud - tp,
            tp / (tp + fp) if tp + fp else None, tp / total_fraud)

drained = (F.col('txn_type').isin('TRANSFER', 'CASH_OUT')
           & (F.col('old_bal_orig') > 0) & (F.col('new_bal_orig') == 0))

candidates = [
    ('DRAINED', drained),
    ('DRAINED_AND_AMT_200K', drained & (F.col('amount') >= 200000)),
    ('DRAINED_AND_DEST_EMPTY', drained & (F.col('old_bal_dest') == 0)),
    ('DRAINED_AND_AMT_AND_DEST_EMPTY',
     drained & (F.col('amount') >= 200000) & (F.col('old_bal_dest') == 0)),
]
display(spark.createDataFrame([score_rule(n, c) for n, c in candidates],
        'rule string, tp long, fp long, fn long, precision double, recall double'))